## 1️⃣ Configurar Runtime com GPU

⚠️ **IMPORTANTE:** Antes de executar, vá em:
- **Runtime > Change runtime type > Hardware accelerator > T4 GPU**
- Ou use TPU se disponível (ainda mais rápido)

In [ ]:
# Verificar se GPU está disponível
!nvidia-smi

## 2️⃣ Upload dos Arquivos

Você precisa fazer upload de **2 arquivos**:
1. `train_classifier.py` (código de treinamento)
2. `dataset_full_texts.json` (dataset com 7778 notícias)

In [ ]:
# Fazer upload dos arquivos
from google.colab import files

print("📤 Faça upload de train_classifier.py")
uploaded = files.upload()

print("\n📤 Faça upload de dataset_full_texts.json")
uploaded = files.upload()

# Criar diretório models
!mkdir -p models
print("\n✅ Arquivos enviados com sucesso!")

## 3️⃣ Instalar Dependências

⚠️ **IMPORTANTE:** Se você reiniciou o runtime após esta célula, **execute ela novamente!**  
O restart limpa toda a memória - os pacotes precisam ser reinstalados.

In [ ]:
# Instalar pacotes necessários
!pip install -q sentence-transformers==5.1.0 xgboost==3.1.2 spacy==3.8.7

# Baixar modelo spaCy para português
!python -m spacy download pt_core_news_sm

print("✅ Dependências instaladas!")
print("⚠️ AGORA reinicie o runtime (Runtime > Restart runtime)")
print("   Depois de reiniciar, execute ESTA CÉLULA NOVAMENTE antes de treinar!")

## 4️⃣ Treinar o Modelo

⚠️ **ANTES DE EXECUTAR:** Certifique-se que executou a célula 3 (dependências) após o restart!

Agora vem a mágica! Com GPU, o treinamento será **5-6x mais rápido**.

In [ ]:
# Verificar se dependências estão instaladas
import sys
try:
    import sentence_transformers
    import xgboost
    import spacy
    print("✅ Dependências OK! Iniciando treinamento...\n")
except ImportError as e:
    print(f"❌ ERRO: Dependências faltando! Execute a célula 3 novamente.")
    print(f"   Pacote faltando: {e.name}")
    sys.exit(1)

# Treinar com todas as melhorias do Gemini
!python train_classifier.py --use_style --style_weight 1.0

## 5️⃣ Download dos Modelos Treinados

Depois que o treinamento terminar, faça download dos arquivos:

In [ ]:
# Listar arquivos gerados
!ls -lh models/

# Criar ZIP com todos os modelos
!zip -r goldenfake_models.zip models/

# Download automático
from google.colab import files
files.download('goldenfake_models.zip')

print("\n✅ Download iniciado! Extraia o ZIP na pasta models/ do seu projeto local.")

## 6️⃣ (Opcional) Visualizar Gráfico de Importância

In [ ]:
# Mostrar o gráfico de feature importance
from IPython.display import Image, display

try:
    display(Image('models/feature_importance.png'))
except:
    print("Gráfico não encontrado. Verifique se o treinamento foi concluído.")

## 📊 Resultado Esperado

Você terá os seguintes arquivos em `goldenfake_models.zip`:

- `classifier.joblib` - Modelo XGBoost treinado
- `label_encoder.joblib` - Encoder de labels (fake/true)
- `style_scaler.joblib` - Scaler das features de estilo
- `config.json` - Configuração do modelo
- `train_texts.json` - Textos de treino (para referência)
- `feature_importance.png` - Gráfico mostrando o que a IA aprendeu

---

## 🔄 De volta ao projeto local

1. Extraia `goldenfake_models.zip` na pasta `models/` do projeto
2. Reinicie o Flask: `python app.py`
3. Teste com a notícia do Augusto Heleno
4. Resultado esperado: **40-70% TRUE** (ao invés de 11.97%)